# Ridge Model (mFRR daily and weekly)

Monthly walk-forward (retrain before each test month on everything already published), one ridge per
`freq × direction`, inputs from `build_features` (+ `add_model_features`). Four variants against the
baseline.

- **Validation** : a validation period before 2025, used to compare and choose.
- **Evaluation** (from 2025): produced once by `uv run python -m src.models.ridge`.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import Ridge

from src.data.data import build_auctions
from src.evaluation import Evaluator
from src.features.build_features import add_model_features, build_features
from src.models.baseline import PreviousKnownAuctionBaseline
from src.models.ridge import FEATURES, model_feature_columns, prepare_auctions, to_contract, walk_forward_ridge


BEST = "background-color: #c6efce; color: #006100; font-weight: bold"   

def _fmt(metric):
    if metric.startswith("pinball"):
        return "{:.4f}"
    if metric.startswith("revenue_share") or metric == "share":
        return "{:.1%}"
    if metric.startswith("revenue"):
        return "{:,.0f}"
    return "{:.3f}"

def _best(values, metric):
    """Green on the best value: lowest error, coverage closest to 0.80, highest revenue."""
    v = pd.to_numeric(pd.Series(values), errors="coerce")
    if metric.startswith("coverage"):
        score = (v - 0.80).abs()
    elif metric.startswith(("mae", "rmse", "pinball")):
        score = v
    else:                                
        score = -v
    return [BEST if s == score.min() else "" for s in score]

def green(table, metric=None, axis=1):
    if metric is not None:
        return table.style.apply(lambda s: _best(s, metric), axis=axis).format(_fmt(metric), na_rep="")
    styled = table.style
    for m in table.columns.get_level_values(0).unique():
        cols = pd.IndexSlice[:, [m]]
        styled = styled.apply(lambda s, m=m: _best(s, m), axis=axis, subset=cols).format(_fmt(m), subset=cols, na_rep="")
    return styled

def green_long(df, metrics, by=("freq", "direction")):
    def styles(d):
        out = pd.DataFrame("", index=d.index, columns=d.columns)
        for m in metrics:
            for idx in d.groupby(list(by)).groups.values():
                out.loc[idx, m] = _best(d.loc[idx, m], m)
        return out
    styled = df.style.apply(styles, axis=None)
    for m in metrics:
        styled = styled.format(_fmt(m), subset=[m], na_rep="")
    return styled

def _gaps(t, metric):
    v = t.apply(pd.to_numeric, errors="coerce")
    if metric.startswith("coverage"):
        d = (v - 0.80).abs()
        return d.sub(d.min(axis=1), axis=0)
    if metric.startswith(("mae", "rmse", "pinball")):
        best = v.min(axis=1)
        return v.sub(best, axis=0).div(best, axis=0)
    best = v.max(axis=1)
    return v.rsub(best, axis=0).div(best, axis=0)

def mean_gap_table(table, metric=None):
    metrics = [metric] if metric is not None else list(table.columns.get_level_values(0).unique())
    out = {}
    for m in metrics:
        t = table if metric is not None else table[m]
        out[m] = _gaps(t, m).mean()
    out = pd.DataFrame(out)
    out.index.name = "mean gap to best"
    fmt = {m: "{:.3f}" if m.startswith("coverage") else "{:.1%}" for m in metrics}
    return out.style.highlight_min(axis=0, props=BEST).format(fmt, na_rep="")

In [2]:
auctions_all = prepare_auctions(build_auctions(product="mFRR"), ("daily", "weekly"))
feats = add_model_features(build_features(auctions_all))     
print("model inputs (all features):", model_feature_columns(feats))
feats.groupby(["freq", "direction"]).size()

model inputs (all features): ['lag1', 'lag2', 'lag3', 'block_00_04', 'block_04_08', 'block_08_12', 'block_12_16', 'block_16_20', 'block_20_24', 'daytype_saturday', 'daytype_sunday_holiday', 'daytype_workday']


freq    direction
daily   downwards    17040
        upwards      17034
weekly  downwards      410
        upwards        410
dtype: int64

## 1. Validation period

Share of auctions without a price (no-buy) per year:

In [3]:
a = auctions_all.copy()
a["year"] = a["gc"].dt.year
a.pivot_table(index="year", columns=["freq", "direction"], values="y",
              aggfunc=lambda y: y.isna().mean()).round(2)

freq          daily            weekly        
direction downwards upwards downwards upwards
year                                         
2018           0.00    0.00       NaN     NaN
2019           0.05    0.00       0.0     0.0
2020           0.00    0.06       0.0     0.0
2021           0.11    0.06       0.0     0.0
2022           0.16    0.10       0.0     0.0
2023           0.19    0.12       0.0     0.0
2024           0.90    0.83       0.0     0.0
2025           0.00    0.00       0.0     0.0
2026           0.03    0.00       0.0     0.0

In [4]:
VAL_START, VAL_LAST_DAY = "2022-01-01", "2024-12-31"
VAL_END = (pd.Timestamp(VAL_LAST_DAY) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")   # exclusive end, for the Evaluator

in_val = (auctions_all["gc"] >= pd.Timestamp(VAL_START, tz="UTC")) & (auctions_all["gc"] < pd.Timestamp(VAL_END, tz="UTC"))
print(VAL_START, "to", VAL_LAST_DAY, ":", in_val.sum(), "auctions,", auctions_all.loc[in_val, "y"].notna().sum(), "with a price")

2022-01-01 to 2024-12-31 : 13064 auctions, 8349 with a price


## 2. Variants vs baseline

**Target:** level **vs** delta

**Features:** no block and day-type one-hots **vs** with block and day-type one-hots

**Groups:** one model per frequency x direction **vs** one model per series

In [5]:
variants = {
    "ridge_lags3":                  dict(feature_cols=FEATURES, target="level"),
    "ridge_lags3_delta":            dict(feature_cols=FEATURES, target="delta"),
    "ridge_allfeats_delta":         dict(feature_cols=model_feature_columns(feats), target="delta"),
    "ridge_lags3_delta_per_series": dict(feature_cols=FEATURES, target="delta", group_cols=["series"]),
}

preds_by_variant = {
    name: walk_forward_ridge(feats, start_test=VAL_START, end_test=VAL_LAST_DAY, **kw)
    for name, kw in variants.items()
}

In [6]:
baseline = PreviousKnownAuctionBaseline().fit(auctions_all)
contracts = {"baseline": baseline.predict(auctions_all[in_val])}
contracts.update({name: to_contract(p, name) for name, p in preds_by_variant.items()})

evaluator = Evaluator(actuals=auctions_all, start_test=VAL_START, end_test=VAL_END)
report = evaluator.evaluate_many(contracts)
mae_table = report.pivot_table(index=["freq", "direction"], columns="model", values="mae")
display(green(mae_table, "mae"))
display(mean_gap_table(mae_table, "mae"))

,mae
mean gap to best,
baseline,5.2%
ridge_allfeats_delta,0.1%
ridge_lags3,4.8%
ridge_lags3_delta,3.9%
ridge_lags3_delta_per_series,3.1%


## 3. Daily, per block
Using `ridge_allfeats_delta`.

In [7]:
VARIANT = "ridge_allfeats_delta"

p = preds_by_variant[VARIANT]
daily = p[(p["freq"] == "daily") & p["y"].notna()]
by_block = pd.DataFrame({
    "baseline": (daily["pred_baseline"] - daily["y"]).abs().groupby([daily["direction"], daily["series"]]).mean(),
    VARIANT: (daily["pred_ridge"] - daily["y"]).abs().groupby([daily["direction"], daily["series"]]).mean(),
    "n": daily.groupby(["direction", "series"]).size(),
})
display(by_block.style.apply(lambda s: _best(s, "mae"), axis=1, subset=["baseline", VARIANT]).format("{:.3f}", subset=["baseline", VARIANT]))
display(mean_gap_table(by_block[["baseline", VARIANT]], "mae"))

,mae
mean gap to best,
baseline,10.4%
ridge_allfeats_delta,2.0%


## 4. Weights learned 

Model trained on everything published before the validation period using `ridge_allfeats_delta`.

In [8]:
cols = model_feature_columns(feats)                                   
tr = feats[feats["avail"] < VAL_START].dropna(subset=cols + ["y"])

rows = []
for (freq, direction), d in tr.groupby(["freq", "direction"]):
    m = Ridge(alpha=1.0).fit(d[cols], d["y"] - d["lag1"])            
    rows.append({"freq": freq, "direction": direction, **dict(zip(cols, m.coef_)), "intercept": m.intercept_})

pd.DataFrame(rows).round(3)

,freq,direction,lag1,lag2,lag3,block_00_04,block_04_08,block_08_12,block_12_16,block_16_20,block_20_24,daytype_saturday,daytype_sunday_holiday,daytype_workday,intercept
0,daily,downwards,-0.515,0.267,0.165,0.016,0.015,-0.011,-0.001,-0.012,-0.007,0.037,0.175,-0.212,0.205
1,daily,upwards,-0.528,0.064,0.243,-0.120,-0.012,0.056,-0.024,0.073,0.027,-0.154,-0.220,0.374,-0.101
2,weekly,downwards,-0.021,0.039,-0.024,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.027
3,weekly,upwards,0.063,-0.117,0.042,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.036


## 5. Quantiles and bidding

In a pay-as-bid auction you are accepted only if your price is at or below the marginal price,
so bidding too high costs everything and bidding too low only part. `pXX_ridge` is a price the actual
marginal price falls below only XX% of the time.

- **coverage** of [p10, p90]: should be close to 80%;
- **share_rejected**: how often the bid is above the actual price (for p10 it should be close to 10%);
- **revenue_share**: revenue of bidding this value in every auction (paid the bid when accepted, nothing
  when rejected), divided by the revenue of bidding exactly the marginal price.

In [9]:
BID_QUANTILES = (0.1, 0.2, 0.3, 0.4, 0.5)
p = walk_forward_ridge(feats, start_test=VAL_START, end_test=VAL_LAST_DAY,
                       quantiles=BID_QUANTILES + (0.9,), **variants[VARIANT])
p = p[p["y"].notna()]
groups = [p["freq"], p["direction"]]

summary = pd.DataFrame({
    "coverage_p10_p90": ((p["y"] >= p["p10_ridge"]) & (p["y"] <= p["p90_ridge"])).groupby(groups).mean(),
    "share_below_p10": (p["y"] < p["p10_ridge"]).groupby(groups).mean(),
})
summary.round(3)

coverage_p10_p90  share_below_p10
freq   direction                                   
daily  downwards             0.844            0.058
       upwards               0.877            0.026
weekly downwards             0.758            0.121
       upwards               0.885            0.057

In [10]:
bids = {"baseline (lag1)": "pred_baseline", "ridge (middle)": "pred_ridge"}
bids.update({f"ridge p{round(100 * q)}": f"p{round(100 * q)}_ridge" for q in BID_QUANTILES})

perfect = (10 ** p["y"]).groupby(groups).sum()
rows = {}
for name, col in bids.items():
    accepted = p[col] <= p["y"]
    revenue = (10 ** p[col]).where(accepted, 0.0)
    rows[(name, "share_rejected")] = (~accepted).groupby(groups).mean()
    rows[(name, "revenue_share")] = revenue.groupby(groups).sum() / perfect

bidding = pd.DataFrame(rows).T
revenue_rows = pd.IndexSlice[pd.IndexSlice[:, "revenue_share"], :]     
display(bidding.style.apply(lambda s: _best(s, "revenue_share"), axis=0, subset=revenue_rows).format("{:.3f}"))
display(mean_gap_table(bidding.xs("revenue_share", level=1).T, "revenue_share"))    

,revenue_share
mean gap to best,
baseline (lag1),50.5%
ridge (middle),33.8%
ridge p10,16.9%
ridge p20,10.4%
ridge p30,9.9%
ridge p40,11.4%
ridge p50,19.5%


## Conclusion

- **Variants.** Four ridge variants against the baseline on the validation period
  2022–2024: level vs delta target, with or without the block / day-type one-hots, one model per group
  vs one per series.
- **Best variant:** `ridge_allfeats_delta` (delta target, with the one-hots). It beats the baseline in the
  daily groups and slightly in weekly upwards, but not in weekly downwards.
- **What it learns.** In price terms, a weighted average of the last three prices (weights adding up to
  0.79–0.99), plus a day-type correction in the daily groups: upward reserve more expensive on workdays, downward
  reserve more expensive on Sundays and holidays. In the weekly groups the predicted change is almost constant,
  so the model is very close to the baseline.
- **Quantiles.** The p10 / p90 come from the training errors, with the same width for every auction of a
  model and month. The intervals are too wide: coverage 0.84–0.89 in three groups (target 0.80), 0.76 in
  weekly downwards.
- **Bidding.** Bidding the middle prediction is rejected about half of the time; a lower quantile earns
  more (best: p40 in the daily groups, p10 in the weekly ones).
- **Kept as the linear reference:** `ridge_allfeats_delta`.